# 03 · Feature Engineering

This notebook builds the **model-ready table** and saves it for notebooks 04–05. Four steps:

1. Clean the GMP columns.
2. Attach each IPO's **listing date** by matching company names against the year files.
3. Build **market-mood features** from the Nifty (return + drawdown around the listing date).
4. Add a few simple **trend features**, fill gaps, and save.

I deliberately keep every step to plain pandas so I can explain each line.


In [1]:
import pandas as pd
import numpy as np
import re
import difflib

PROC = "../data/processed"
target = "listing_gain_close_pct"

df       = pd.read_csv(f"{PROC}/ipo_main.csv")
listings = pd.read_csv(f"{PROC}/listings.csv", parse_dates=["list_date"])
nifty    = pd.read_csv(f"{PROC}/nifty.csv", parse_dates=["Date"])
print("ipo:", df.shape, "| listings:", len(listings), "| nifty:", len(nifty))


ipo: (312, 39) | listings: 325 | nifty: 1097


## 1. Clean GMP

Recompute the GMP percent from the (more reliable) rupee value, and drop rows with an impossible
>200% reading. Same logic as the EDA notebook — here it is part of the official pipeline.


In [2]:
for d in range(1, 6):
    df[f"gmp_day{d}_pct"] = (df[f"gmp_day{d}_rs"] / df["issue_price"] * 100).round(2)

bad = pd.Series(False, index=df.index)
for d in range(1, 6):
    bad = bad | (df[f"gmp_day{d}_pct"] > 200)
df = df[~bad].reset_index(drop=True)
print("rows after GMP clean:", len(df))


rows after GMP clean: 309


## 2. Attach the listing date by name matching

The two files spell company names differently (`Ltd.` vs nothing, `&` vs `and`, brackets, etc.). So I
**normalise each name into a plain key** — lowercase, drop suffixes and punctuation — and match on
that. If the exact key is missing, `difflib.get_close_matches` finds the closest one.


In [3]:
def clean_name(name):
    name = str(name).lower().replace("&", "and")
    name = re.sub(r"\(.*?\)", "", name)     # remove anything in brackets
    name = re.sub(r"ipo$", "", name)          # remove a trailing 'ipo'
    for w in ["ltd", "limited", "the", "co", "pvt", "private", "inc"]:
        name = re.sub(r"\b" + w + r"\b", "", name)
    return re.sub(r"[^a-z0-9]", "", name)     # keep only letters and numbers

listings["key"] = listings["Company Name"].apply(clean_name)
date_map = dict(zip(listings["key"], listings["list_date"]))
all_keys = list(date_map)

def find_date(company):
    k = clean_name(company)
    if k in date_map:                          # exact key match
        return date_map[k]
    close = difflib.get_close_matches(k, all_keys, n=1, cutoff=0.85)   # else closest name
    return date_map[close[0]] if close else pd.NaT

df["list_date"] = df["company"].apply(find_date)
print("matched by name:", df["list_date"].notna().sum(), "/", len(df))


matched by name: 307 / 309


In [4]:
# two very recent IPOs are listed under a different legal name in the year files,
# so I fill their dates by hand from the public exchange record
manual = {"KisshtIPO": "2026-05-08", "Aeroplane Basmati RiceIPO": "2026-04-02"}
for company, d in manual.items():
    df.loc[df["company"] == company, "list_date"] = pd.Timestamp(d)

print("matched after manual fill:", df["list_date"].notna().sum(), "/", len(df))


matched after manual fill: 309 / 309


## 3. Market-mood features (Nifty 50)

Listing gains depend a lot on how the broad market feels. From the Nifty I build three features:

- **5-day return** and **1-month return** — short-term momentum into the listing.
- **drawdown** — how far below its all-time high the index is (the bull-vs-tired-market regime).

`pct_change(n)` gives the return over *n* rows; `cummax()` gives the running peak for the drawdown.


In [5]:
nifty = nifty.sort_values("Date").reset_index(drop=True)
nifty["nifty_ret_5d"]    = nifty["Close"].pct_change(5)  * 100
nifty["nifty_ret_1m"]    = nifty["Close"].pct_change(22) * 100      # ~22 trading days in a month
nifty["nifty_drawdown"]  = (nifty["Close"] / nifty["Close"].cummax() - 1) * 100
nifty[["Date", "Close", "nifty_ret_5d", "nifty_ret_1m", "nifty_drawdown"]].tail()


,Date,Close,nifty_ret_5d,nifty_ret_1m,nifty_drawdown
1092,2026-06-03,23405.60,-2.124723,-2.466710,-11.101827
1093,2026-06-04,23416.55,-2.052106,-2.913642,-11.060237
1094,2026-06-05,23366.70,-0.768863,-2.771629,-11.249575
1095,2026-06-08,23123.00,-1.110227,-4.964664,-12.175186
1096,2026-06-09,23242.10,-1.028167,-4.458279,-11.722826


To attach the right Nifty row to each IPO I use **`pd.merge_asof`** — the standard pandas tool for
an "as-of" join. For each listing date it grabs the **most recent Nifty trading day on or before** it
(`direction="backward"`), which handles weekends and holidays automatically. Both tables must be
sorted by the join date.


In [6]:
nifty_feats = ["nifty_ret_5d", "nifty_ret_1m", "nifty_drawdown"]

dated = df.dropna(subset=["list_date"]).sort_values("list_date")
merged = pd.merge_asof(
    dated,
    nifty[["Date"] + nifty_feats],
    left_on="list_date", right_on="Date",
    direction="backward",
)
# write the looked-up values back onto df, lined up by original row index
merged.index = dated.index
for c in nifty_feats:
    df.loc[merged.index, c] = merged[c]

print("nifty features filled:", df["nifty_ret_1m"].notna().sum(), "/", len(df))
df[["company", "list_date"] + nifty_feats].dropna().head()


nifty features filled: 308 / 309


,company,list_date,nifty_ret_5d,nifty_ret_1m,nifty_drawdown
0,Abans HoldingsIPO,2022-12-23,-2.529969,-2.520631,-5.345914
1,Adani WilmarIPO,2022-02-08,-1.764252,-2.700060,-5.687920
2,Aether IndustriesIPO,2022-06-03,1.417830,-0.559433,-9.415505
4,Archean Chemical IndustriesIPO,2022-11-21,-0.923120,3.848584,-1.356354
5,Bikaji Foods InternationalIPO,2022-11-16,1.391474,8.200725,0.000000


## 4. Trend features + final table

Three small features that capture *movement*, not just the level:

- **`gmp_trend`** — is the grey market heating up or cooling over the 5 days?
- **`sub_momentum`** — how fast the subscription book is building (day-2 vs day-1).
- **`qib_vs_retail`** — big institutional investors vs small retail investors.

I **drop the fundamentals** (PE, EBITDA, debt) — the EDA showed they barely move the listing gain. Any
remaining gaps are filled with the column median so I keep every row.


In [7]:
df["gmp_trend"]     = df["gmp_day5_pct"] - df["gmp_day1_pct"]
df["sub_momentum"]  = df["sub_day2_x"] / (df["sub_day1_x"] + 0.1)
df["qib_vs_retail"] = df["sub_qib_x"]  / (df["sub_retail_x"] + 0.1)

features = ["issue_size_cr",
            "sub_day1_x", "sub_day2_x", "sub_day3_x",
            "total_sub_x", "sub_qib_x", "sub_hni_x", "sub_retail_x", "sub_overall_x",
            "gmp_day1_pct", "gmp_day2_pct", "gmp_day3_pct", "gmp_day4_pct", "gmp_day5_pct",
            "nifty_ret_5d", "nifty_ret_1m", "nifty_drawdown",
            "gmp_trend", "sub_momentum", "qib_vs_retail"]

for c in features:
    df[c] = df[c].fillna(df[c].median())

print(len(features), "features")


20 features


In [8]:
# keep only what the models need, and save
keep = ["company", "year", "list_date", target] + features
out = df[keep].copy()
out.to_csv(f"{PROC}/ipo_features.csv", index=False)
print("saved ipo_features.csv  shape:", out.shape)
out.head()


saved ipo_features.csv  shape: (309, 24)


,company,year,list_date,listing_gain_close_pct,issue_size_cr,sub_day1_x,sub_day2_x,sub_day3_x,total_sub_x,sub_qib_x,...,gmp_day2_pct,gmp_day3_pct,gmp_day4_pct,gmp_day5_pct,nifty_ret_5d,nifty_ret_1m,nifty_drawdown,gmp_trend,sub_momentum,qib_vs_retail
0,Abans HoldingsIPO,2022,2022-12-23,-4.44,360.11,1.16,4.93,80.80,80.80,117.33,...,6.30,4.44,5.56,1.85,-2.529969,-2.520631,-5.345914,-4.45,3.912698,3.928021
1,Adani WilmarIPO,2022,2022-02-08,16.63,410.05,13.32,34.85,182.44,182.44,331.60,...,26.09,21.74,19.57,15.22,-1.764252,-2.700060,-5.687920,-6.52,2.596870,5.191796
2,Aether IndustriesIPO,2022,2022-06-03,20.62,144.89,8.53,29.46,148.75,148.75,87.21,...,0.78,0.78,1.56,0.00,1.417830,-0.559433,-9.415505,-1.56,3.413673,0.434984
3,AGS Transact TechnologiesIPO,2022,2022-01-31,-12.00,260.15,17.71,54.55,227.51,227.51,242.40,...,11.43,17.14,11.43,11.43,-1.574034,0.754293,-5.288643,0.00,3.062886,2.385357
4,Archean Chemical IndustriesIPO,2022,2022-11-21,21.13,12500.00,0.35,1.11,16.69,16.69,55.47,...,14.74,14.74,17.20,18.43,-0.923120,3.848584,-1.356354,3.69,2.466667,36.735099


The model-ready table is saved. Notebook 04 trains on it; notebook 05 makes the final predictions
and the GMP comparison.
